# Energy Transition & Electric Mobility in Portugal - Data Extraction

**Phase 1: Extraction**

Pulls REN daily production/consumption data and E-REDES electric mobility
consumption data (aggregated by year/month/district/municipality).

In [1]:
import requests
import json
import time
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path

# VS Code Jupyter runs notebooks with the working directory set to the folder
# containing the .ipynb file. This notebook lives in notebooks/, so the
# project root is exactly one level up — no marker file, no searching.
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

REN_BASE_URL = "https://servicebus.ren.pt/datahubapi/electricity"
EREDES_BASE_URL = "https://e-redes.opendatasoft.com/api/explore/v2.1/catalog/datasets"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data will be saved to: {DATA_RAW}")

Project root: c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics
Data will be saved to: c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\raw


## REN - daily production/consumption breakdown

A single call per day already returns every source (Hydro, Solar, Wind,
Natural Gas, Biomass, Coal, Wave, Import, Export, batteries, etc.) as
separate series, at 15-minute resolution.

In [2]:
def fetch_ren_daily_breakdown(target_date, delay=0.5):
    """Fetch REN electricity production breakdown for a single date."""
    params = {"culture": "en-US", "date": target_date.isoformat()}
    response = requests.get(
        f"{REN_BASE_URL}/ElectricityProductionBreakdownDaily", params=params
    )
    time.sleep(delay)

    if response.status_code != 200:
        print(f"Failed for {target_date}: {response.status_code}")
        return None

    return response.json()


def build_date_range(start_date, end_date):
    days = (end_date - start_date).days
    return [start_date + timedelta(days=i) for i in range(days + 1)]


END_DATE = date.today() - timedelta(days=1)
START_DATE = END_DATE - relativedelta(years=3)
date_range = build_date_range(START_DATE, END_DATE)

print(f"Fetching {len(date_range)} days ({START_DATE} to {END_DATE})...")

ren_raw_results = []
ren_failed_dates = []

for d in date_range:
    result = fetch_ren_daily_breakdown(d)
    if result is not None:
        ren_raw_results.append({"date": d, "data": result})
    else:
        ren_failed_dates.append(d)

print(f"Success: {len(ren_raw_results)} | Failed: {len(ren_failed_dates)}")

output_path = DATA_RAW / "ren_production_breakdown_raw.json"
with open(output_path, "w") as f:
    json.dump(ren_raw_results, f, indent=2, default=str)

print(f"Saved to {output_path}")

Fetching 1097 days (2023-10-06 to 2026-10-06)...


KeyboardInterrupt: 

## E-REDES - electric mobility consumption, aggregated

Two datasets cover all 18 mainland districts between them:
`_01_a_10` and `_11_a_18`. Aggregated server-side (sum of energy per
year/month/district/municipality) instead of pulling raw hourly rows.

`total_count` is unreliable on `group_by` queries here - pagination stops
when a page comes back empty, not by trusting `total_count`.

In [7]:
def fetch_eredes_grouped(dataset_id, page_size=200):
    """Fetch all grouped/aggregated records, paginating until an empty page."""
    all_results = []
    offset = 0

    while True:
        response = requests.get(
            f"{EREDES_BASE_URL}/{dataset_id}/records",
            params={
                "select": "sum(energia_consumida) as total_energia_kwh",
                "group_by": "ano,mes,distrito,concelho",
                "limit": page_size,
                "offset": offset,
            },
        )
        response.raise_for_status()
        results = response.json()["results"]

        if not results:
            break

        all_results.extend(results)
        offset += page_size
        time.sleep(0.3)

    return all_results


EREDES_DATASETS = [
    "consumo_horario_mobilidade_eletrica_01_a_10",
    "consumo_horario_mobilidade_eletrica_11_a_18",
]

eredes_raw_results = []
for dataset_id in EREDES_DATASETS:
    print(f"Fetching {dataset_id}...")
    rows = fetch_eredes_grouped(dataset_id)
    print(f"  {len(rows)} grouped rows")
    eredes_raw_results.extend(rows)

print(f"Total rows: {len(eredes_raw_results)}")

output_path = DATA_RAW / "eredes_mobility_grouped_raw.json"
with open(output_path, "w") as f:
    json.dump(eredes_raw_results, f, indent=2, default=str)

print(f"Saved to {output_path}")

Fetching consumo_horario_mobilidade_eletrica_01_a_10...
  1764 grouped rows
Fetching consumo_horario_mobilidade_eletrica_11_a_18...
  1572 grouped rows
Total rows: 3336
Saved to c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\raw\eredes_mobility_grouped_raw.json


## Extraction: REN market prices

Returning to Phase 1 to fill the gap left for the price correlation analysis.
Same pattern as production breakdown - explore structure first, then extract
the full date range.

In [8]:
test_params = {"culture": "en-US", "date": "2025-01-25"}
response = requests.get(
    f"{REN_BASE_URL}/ElectricityMarketPricesDaily", params=test_params
)
response.raise_for_status()

price_data = response.json()
print(json.dumps(price_data, indent=2)[:1500])

{
  "yAxis": {
    "tickInterval": 10,
    "title": {
      "text": "\u20ac/MWh"
    }
  },
  "xAxis": {
    "max": 24,
    "tickInterval": 1,
    "categories": [
      "1",
      "2",
      "3",
      "4",
      "5",
      "6",
      "7",
      "8",
      "9",
      "10",
      "11",
      "12",
      "13",
      "14",
      "15",
      "16",
      "17",
      "18",
      "19",
      "20",
      "21",
      "22",
      "23",
      "24"
    ],
    "title": {
      "text": "Hour"
    },
    "export": [
      "1",
      "2",
      "3",
      "4",
      "5",
      "6",
      "7",
      "8",
      "9",
      "10",
      "11",
      "12",
      "13",
      "14",
      "15",
      "16",
      "17",
      "18",
      "19",
      "20",
      "21",
      "22",
      "23",
      "24"
    ]
  },
  "series": [
    {
      "data": [
        57.03,
        35,
        25.2,
        19.71,
        16.8,
        17.72,
        25,
        49.29,
        37.7,
        33.01,
        24.08,
        45.0

In [9]:
print("Series included:")
for s in price_data["series"]:
    print(f"  - {s['name']} ({len(s['data'])} points)")

Series included:
  - PT (24 points)
  - ES (24 points)


In [10]:
def fetch_ren_market_prices(target_date, delay=0.5):
    """Fetch REN/MIBEL market prices (PT and ES) for a single date."""
    params = {"culture": "en-US", "date": target_date.isoformat()}
    response = requests.get(
        f"{REN_BASE_URL}/ElectricityMarketPricesDaily", params=params
    )
    time.sleep(delay)

    if response.status_code != 200:
        print(f"Failed for {target_date}: {response.status_code}")
        return None

    return response.json()


# Redefined here to be safe and self-contained, rather than assuming the
# earlier date_range variable is still the one we want
END_DATE = date.today() - timedelta(days=1)
START_DATE = END_DATE - relativedelta(years=3)
price_date_range = build_date_range(START_DATE, END_DATE)

print(
    f"Fetching {len(price_date_range)} days of market prices ({START_DATE} to {END_DATE})..."
)

market_price_raw = []
failed_price_dates = []

for d in price_date_range:
    result = fetch_ren_market_prices(d)
    if result is not None:
        market_price_raw.append({"date": d, "data": result})
    else:
        failed_price_dates.append(d)

print(f"Success: {len(market_price_raw)} | Failed: {len(failed_price_dates)}")

output_path = DATA_RAW / "ren_market_prices_raw.json"
with open(output_path, "w") as f:
    json.dump(market_price_raw, f, indent=2, default=str)

print(f"Saved to {output_path}")

Fetching 1097 days of market prices (2023-09-30 to 2026-09-30)...
Failed for 2026-06-30: 404
Failed for 2026-07-01: 404
Failed for 2026-07-02: 404
Failed for 2026-07-03: 404
Failed for 2026-07-04: 404
Failed for 2026-07-05: 404
Failed for 2026-07-06: 404
Failed for 2026-07-07: 404
Failed for 2026-07-08: 404
Failed for 2026-07-09: 404
Failed for 2026-07-10: 404
Failed for 2026-07-11: 404
Failed for 2026-07-12: 404
Failed for 2026-07-13: 404
Failed for 2026-07-14: 404
Failed for 2026-07-15: 404
Failed for 2026-07-16: 404
Failed for 2026-07-17: 404
Failed for 2026-07-18: 404
Failed for 2026-07-19: 404
Failed for 2026-07-20: 404
Failed for 2026-07-21: 404
Failed for 2026-07-22: 404
Failed for 2026-07-23: 404
Failed for 2026-07-24: 404
Failed for 2026-07-25: 404
Failed for 2026-07-26: 404
Failed for 2026-07-27: 404
Failed for 2026-07-28: 404
Failed for 2026-07-29: 404
Failed for 2026-07-30: 404
Failed for 2026-07-31: 404
Failed for 2026-08-01: 404
Failed for 2026-08-02: 404
Failed for 2026-

## Next: 02_cleaning.ipynb